<a href="https://colab.research.google.com/github/JakeOh/202511_BD53/blob/main/lab_ml/ml22_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EXAONE 모델을 사용한 질문-답변 시스템

현재 Colab에 설치된 transformers 패키지의 버전은 5.0.0.

EXAONE 모델을 사용하기 위해서는 5.1.0 버전이 필요.

In [2]:
!pip install -U transformers==5.1.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 131.0 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


# Imports

In [3]:
import numpy as np
import scipy
import transformers

In [4]:
transformers.__version__

'5.1.0'

In [5]:
# 깃허브에 ipynb 파일을 업로드할 때 다운로드 상태(진행바) 표시줄 때문에 오류 발생
# -> 깃허브에 정상적으로 업로드되게 하기 위해서
transformers.utils.logging.disable_progress_bar()

# EXAONE-3.5 모델

In [6]:
pipe = transformers.pipeline(
    task='text-generation',
    model='LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct',
    device=0,  # GPU 사용
    trust_remote_code=True
)

A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- configuration_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct:
- modeling_exaone.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


In [7]:
# 메시지 템플릿: 'role'과 'content'를 키로 갖는 dict들을 아이템으로 갖는 리스트.
# role: 역할(system, user, assistant)
# content: 내용
messages = [
    {
        'role': 'system',
        'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야. \
        확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는 \
        간단하고 친절한 답변을 생성해줘.'
    },
    {
        'role': 'user',
        'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'
    }
]

In [8]:
result = pipe(messages, max_new_tokens=200)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [9]:
print(result)
#> pipeline 호출 결과: 'generated_text' 키를 갖는 dict 1개를 저장하고 있는 리스트

[{'generated_text': [{'role': 'system', 'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'}, {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'}, {'role': 'assistant', 'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 포함되어 있어 편리하게 사용하실 수 있도록 준비되었습니다. 하지만 정확한 날짜 확인을 위해서는 제품 담당자님께 연락하시는 것이 가장 정확한 정보를 얻으실 수 있을 것 같아요. 담당자님께 문의하시면 바로 답변드리겠습니다!'}]}]


In [10]:
len(result)

1

In [11]:
result[0]  #> dict

{'generated_text': [{'role': 'system',
   'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
  {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
  {'role': 'assistant',
   'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 포함되어 있어 편리하게 사용하실 수 있도록 준비되었습니다. 하지만 정확한 날짜 확인을 위해서는 제품 담당자님께 연락하시는 것이 가장 정확한 정보를 얻으실 수 있을 것 같아요. 담당자님께 문의하시면 바로 답변드리겠습니다!'}]}

In [12]:
result[0].keys()

dict_keys(['generated_text'])

result - `[ { 'generated_text': [...] } ]`

In [13]:
result[0]['generated_text']
#> 'role'과 'content'를 키로 갖는 dict들의 list.
#> 3개의 dict: 첫 2개는 입력값, 마지막 1개 AI가 생성한 답변

[{'role': 'system',
  'content': '너는 쇼핑몰 Q&A에 올라온 질문에 답변하는 챗봇이야.         확정적인 답변을 하지 말고, 제품 담당자가 정확한 답변을 하기 위해 시간이 필요하다는         간단하고 친절한 답변을 생성해줘.'},
 {'role': 'user', 'content': '이 다이어리에는 내년의 공휴일이 표기되어 있나요?'},
 {'role': 'assistant',
  'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 포함되어 있어 편리하게 사용하실 수 있도록 준비되었습니다. 하지만 정확한 날짜 확인을 위해서는 제품 담당자님께 연락하시는 것이 가장 정확한 정보를 얻으실 수 있을 것 같아요. 담당자님께 문의하시면 바로 답변드리겠습니다!'}]

In [14]:
result[0]['generated_text'][-1]

{'role': 'assistant',
 'content': '네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 포함되어 있어 편리하게 사용하실 수 있도록 준비되었습니다. 하지만 정확한 날짜 확인을 위해서는 제품 담당자님께 연락하시는 것이 가장 정확한 정보를 얻으실 수 있을 것 같아요. 담당자님께 문의하시면 바로 답변드리겠습니다!'}

In [15]:
result[0]['generated_text'][-1]['content']

'네, 맞습니다! 저희 다이어리에는 내년의 공휴일 정보가 포함되어 있어 편리하게 사용하실 수 있도록 준비되었습니다. 하지만 정확한 날짜 확인을 위해서는 제품 담당자님께 연락하시는 것이 가장 정확한 정보를 얻으실 수 있을 것 같아요. 담당자님께 문의하시면 바로 답변드리겠습니다!'

pipeline은 실행할 때마다 다른 답변(텍스트)를 생성함.

pipeline의 파라미터:

*   `return_full_text`: 이전의 대화 기록을 모두 리턴할 지 여부를 설정. 기본값은 True.
*   `do_sample`: 샘플링 전략을 사용할 지 여부를 설정. 기본값은 True.
    *   `do_sample=True`: 메시지 프롬프트가 같아도 실행할 때마다 생성되는 텍스트가 달라짐.
    *   `do_sample=False`
        *   메시지 프롬프트가 같으면 항상 같은 텍스트가 생성됨.
        *   가장 확률이 높은 토큰들만 선택해서 텍스트를 생성.
*   샘플링 전략
    *   temperature(온도)
    *   top-k 방식
    *   top-p 방식


In [16]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              do_sample=False)
print(result)
#> result - [ { 'generated_text': '답변' } ]
#> 실행할 때마다 항상 같은 답변.

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요. 제품 담당자분께서 가장 정확한 정보를 제공해 드릴 수 있을 것 같아요. 그분께서는 다이어리의 최신 업데이트 내용과 내년 공휴일 정보를 확인하실 수 있으니, 조금 더 자세히 알아보시려면 연락을 취해보시는 건 어떨까요? 친절하게 답변해 드리겠습니다! 😊'}]


# 샘플링 전략 - temperature

In [17]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=10.0)
print(result)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': ' 맞아 여쭤 보긴 하네![response custom content][Sure] 고객분 thanks asking thoughtfulQuesiton이구 진짜 자세해요네, 저에세  물론내년 calendarspecific Information이나 기념일은 최신 날짜 표시 목록서 다룰수 upto있잖아요✨. 근데 구체성 보다은 바로 최신 정책은 담당님 팀에 상의 부탁드립니다 저희가 직접 해당 날짜를 정확히 판별 못하거aders팀이가 완벽체크 할테인답 해줄 텐데, 쉽겟자 곧 말씀 나누시기 바랬더이라네요그럼 조금이라도 기다려야죠 ~ 기다려 보면 훨씬 더 자세히 도움이 풀릴 듯한 확신을 드리게요][Encapsulator]{답변 자동 작성 시도 취소 (해당 작업에 있도록 답변 완료한 시점에는 적합하려 시도 중지 중이나 이 문장 내 작업 지속적으로 이어지네요).} 고객 요청 완벽 존중 하고 빠르기에 대한 존중도 충분히 인정되게 안내 드리죠 🩃 어떤 것 이상이셔도 물어마시려면 지금 이 채널 연락 필요하므로 천천히 고민보거나 문의 부탁드립니다. ['}]


In [18]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=0.001)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 미리 표시되어 있는지에 대해 궁금하시군요. 제품 담당자분께서 가장 정확한 정보를 제공해 드릴 수 있을 것 같아요. 그분께서는 다이어리의 최신 업데이트 내용과 내년 공휴일 정보를 확인하실 수 있으니, 조금 더 자세히 알아보시려면 연락을 취해보시는 건 어떨까요? 친절하게 답변해 드리겠습니다! 😊'}]


temperature(온도)를 1보다 큰 값을 사용하면 문장을 생성할 때 선택되는 토큰들이 다양해 지기 때문에 답변이 무작위짐.

temperature를 1보다 작은 값을 사용하면 높은 확률을 가진 토큰들이 선택될 가능성이 더 커짐. 생성되는 답변이 더 일관되어 짐.

In [19]:
# 동전 던지기
probs = [0.5, 0.5]  # 앞면 확률, 뒷면 확률

np.random.multinomial(n=100, pvals=probs)  # 동전 던지기 100번 실험

array([55, 45])

In [20]:
# 주사위 던지기
probs = [1/6] * 6
np.random.multinomial(n=600, pvals=probs)

array([105,  93, 108, 109,  98,  87])

logit(로짓): softmax 함수의 아규먼트. LLM에서는 어휘 사전에 포함된 각 토큰에 대한 점수.

In [21]:
logits = [1, 2, 3, 4, 10]  # 5개 토큰의 점수
probs = scipy.special.softmax(logits)
print(probs)

[1.22936559e-04 3.34176214e-04 9.08385131e-04 2.46924679e-03
 9.96165255e-01]


In [22]:
np.random.multinomial(n=200, pvals=probs)

array([  0,   0,   0,   0, 200])

In [23]:
logits = np.array([1, 2, 3, 4, 100])
probs = scipy.special.softmax(logits)
print(probs)

[1.01122149e-43 2.74878501e-43 7.47197234e-43 2.03109266e-42
 1.00000000e+00]


In [24]:
np.random.multinomial(n=200, pvals=probs)

array([  0,   0,   0,   0, 200])

In [25]:
probs = scipy.special.softmax(logits / 10)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[5.01629119e-05 5.54385914e-05 6.12691190e-05 6.77128484e-05
 9.99765417e-01]


array([  0,   0,   0,   0, 200])

In [26]:
probs = scipy.special.softmax(logits / 100)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[0.14810557 0.14959406 0.1510975  0.15261606 0.39858682]


array([37, 29, 34, 29, 71])

In [27]:
probs = scipy.special.softmax(logits / 0.1)
print(probs)
np.random.multinomial(n=200, pvals=probs)

[0. 0. 0. 0. 1.]


array([  0,   0,   0,   0, 200])

*   logit 값들을 1보다 큰 수로 나눈 후 softmax 함수로 확률을 계산하면, logit 값이 작은 토큰들의 확률이 커지는 효과.
    *   토큰을 선택하는 다양성이 커짐.
    *   생성되는 텍스트가 다양해짐.
*   logit 값을 1보다 작은 수로 나눈 후 softmax 함수로 확률을 계산하면, logit 값이 큰 토큰들의 확률을 더 크게 만드는 효과
    *   토큰을 선택하는 다양성이 작아짐.
    *   거의 비슷한 텍스트들이 생성됨.
*   LLM에서 온도(temperature)는 logit 값들을 나눠주는 수.

# top-k 샘플링

모델이 계산한 logit(점수) 값들을 정렬해서 최상위 k개의 토큰들 중에서 선택하는 방법.

k값이 클 수록 더 다양한 텍스트들이 생성될 수 있음.

In [30]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=1.5,
              top_k=100)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '아니요, 바로 질문해 주셔서 감사합니다! 하지만 자세히 알아보기까지 조금 시간이 필요합니다. 제품담당팀에서 정확한 정보를 제공하기 위해 고민 중인데, 일반적으로 다이어리의 공휴일 표기 여부는 제품의 디자인과 구매처에 따라 다르기 마련예요. 정확한 답변을 위해 잠시 체크하고elia요. 곧 당신께 소식을 전해드릴게요! 언제든지 궁금한 점이 다시 있으시다면 편하게 물어주사요.'}]


In [32]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=1.5,
              top_k=10)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '아뇨, 현재 제공되는 정보에 따르면 이 다이어리에 특별히 내년의 공휴일 표기가 포함되어 있는 것 같지 않습니다. 다이어리 구매 시에 정확한 필요에 맞도록 확인하시거나, 판매 시점에 직원님께 직접 문의하시는 것이 가장 정확한 답변을 얻는 방법일 것 같습니다. 도움이 되셨길 바랍니다! 😊'}]


# top-p 샘플링

모델이 출력한 logit 값으로 계산한 확률의 크기 순으로 정렬했을 때 누적 확률 p까지의 토큰을 선택하는 방법.

top_p(누적 확률) 값이 클 수록 다양한 텍스트, 작을 수록 일관된 텍스트들이 생성됨.

누적 확률 p가 동일하더라도 선택될 수 있는 토큰의 개수가 달라질 수 있음.

top-p 샘플링은 다양한 확률 분포에 유연하게 대처할 수 있음.

top-k 샘플링은 선택하는 토큰들의 개수를 고정하기 때문에, 높은 확률을 가진 토큰이 선택에서 제외될 가능성이 있음.

In [34]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=1.5,
              top_p=0.9)
print(result)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '죄송합니다만, 제가 직접 해당 다이어리의 정보를 확인하거나 특정 제품 세부 사항을 답변드릴 수는 없어요. 정확한 내용은 해당 제품을 판매하고 있는 쇼핑몰의 고객 서비스나 제품 페이지에서 확인하실 수 있을 것 같습니다. 직원에게 문의하시면 가장 빠르고 정확한 답변을 받으실 수 있을 거예요! 힘내세요!'}]


In [36]:
result = pipe(messages,
              max_new_tokens=200,
              return_full_text=False,
              temperature=1.5,
              top_p=0.3)
print(result)

Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': '안녕하세요! 다이어리에 내년의 공휴일이 포함되어 있는지에 대해 자세히 알려드리기 위해서는 제품 담당자님께 문의하시는 것이 가장 정확할 것 같아요. 현재로선 확실한 정보를 제공하기 어렵지만, 담당자분께서는 다이어리의 정확한 내용과 업데이트 계획에 대해 자세히 알려드릴 수 있을 거예요. 궁금한 점이 더 있으시다면 언제든지 다시 물어봐 주세요!'}]
